# 01 — Data Cleaning & Integration Pipeline
**Deliverable A | Qiyas AI Hackathon | Team Quatro**

This notebook establishes an automated, leakage-free data engineering pipeline that converts three messy raw exports into a standardized, time-aligned, feature-engineered master modeling frame.

### Outline:
1. **A1. Raw Data Inspection & Issue Identification**
2. **A2. Time & Key Standardization (The Clock Proof)**
3. **A3. Multi-Table Integration Map & Joining Architecture**
4. **A4. Join Audit & Match Quality**
5. **A5. Concrete Join Proofs (3 Case Studies)**
6. **A6. Feature Engineering Dictionary**
7. **A7. Automated Data Integrity Checks**
8. **A8. Export Master Tables**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cleaning import load_ride_demand, parse_pickup_hour, canonicalize_zone, CANONICAL_ZONES, ZONE_ALIASES
from integration import load_weather, load_events, add_weather_features, add_event_features
from features import add_calendar_features, full_features

## A1. Raw Data Inspection & Issue Identification
We load the three raw CSV files and identify the core data quality issues across all tables.

In [ ]:
raw_trips = pd.read_csv("../data/raw/ride_demand_train.csv")
raw_weather = pd.read_csv("../data/raw/weather_hourly.csv")
raw_events = pd.read_csv("../data/raw/events_calendar.csv")

print(f"Raw Trips shape    : {raw_trips.shape}")
print(f"Raw Weather shape  : {raw_weather.shape}")
print(f"Raw Events shape   : {raw_events.shape}")

### The Cleaning Log
Summary of the issues detected and fixes applied across the three tables:

In [ ]:
cleaning_log = pd.DataFrame([
    {"Table": "Trips", "Columns": "avg_fare_birr, avg_wait_min, active_drivers", "Issue": "Target Leakage", "Fix": "Dropped before feature engineering", "Rationale": "Unknown at forecast time (Rule 6)"},
    {"Table": "Trips", "Columns": "zone", "Issue": "Inconsistent casing & aliases", "Fix": "Mapped 8 aliases to 12 canonical zones", "Rationale": "Uniform spatial keys"},
    {"Table": "Trips", "Columns": "pickup_hour", "Issue": "Mixed date formats (ISO vs DMY)", "Fix": "Regex format router", "Rationale": "Prevents month/day transposition"},
    {"Table": "Trips", "Columns": "trips", "Issue": "Negative & NaN targets", "Fix": "Filtered trips >= 0", "Rationale": "Removes sensor errors"},
    {"Table": "Trips", "Columns": "[zone, pickup_hour]", "Issue": "Duplicate timestamps", "Fix": "Grouped and averaged", "Rationale": "Ensures unique zone-hour key"},
    {"Table": "Weather", "Columns": "temp_c", "Issue": "Fahrenheit values (>45°C)", "Fix": "Converted (F-32)*5/9 to Celsius", "Rationale": "Restores physical temperature scale"},
    {"Table": "Weather", "Columns": "rain_mm", "Issue": "Negative sentinels (-9999.0)", "Fix": "Masked to 0.0", "Rationale": "Removes invalid sensor dropouts"},
    {"Table": "Weather", "Columns": "timestamp", "Issue": "UTC clock mismatch", "Fix": "Converted UTC to EAT (+3h)", "Rationale": "Aligns weather with Addis local trip time"},
    {"Table": "Events", "Columns": "zone", "Issue": "Composite & citywide labels", "Fix": "Exploded to affected canonical zones", "Rationale": "Matches multi-zone events accurately"},
    {"Table": "Events", "Columns": "end_datetime", "Issue": "Missing or negative durations", "Fix": "Imputed with median duration by type", "Rationale": "Preserves valid confirmed events"},
    {"Table": "Events", "Columns": "status", "Issue": "Cancelled events", "Fix": "Filtered status == 'confirmed'", "Rationale": "Cancelled events leave no demand footprint"}
])
display(cleaning_log)

## A2. Time & Key Standardization & The Clock Proof
### (a) Canonicalizing Zones
All zone variations are standardized into exactly 12 canonical Addis Ababa zones.

In [ ]:
print("Canonical 12 Zones:")
for i, z in enumerate(CANONICAL_ZONES, 1):
    print(f"  {i}. {z}")

print("\nSample Alias Mappings:")
for k, v in list(ZONE_ALIASES.items())[:5]:
    print(f"  '{k}' -> '{v}'")

### (b) Parsing Timestamp Formats Without Misreading
Demonstrating regex-routed date parsing to prevent day-first vs month-first corruption:

In [ ]:
sample_dates = pd.Series([
    "2025-01-09 18:00",           # Jan 9, 2025
    "16/08/2025 04:00",           # Aug 16, 2025 (day-first)
    "2025-05-27T23:00:00+03:00",   # May 27, 2025 (offset)
])
parsed_samples = parse_pickup_hour(sample_dates)
for raw_d, p_d in zip(sample_dates, parsed_samples):
    print(f"Raw: '{raw_d}' -> Parsed: {p_d}")

### (c) Clock Proof: Proving Weather Timestamps are UTC
Addis Ababa operates on East Africa Time (EAT = UTC+3). We prove that shifting weather timestamps by +3 hours aligns peak temperature with local 14:00 solar peak.

In [ ]:
raw_weather_df = raw_weather.copy()
# Raw UTC peak vs Local EAT peak
raw_weather_df['ts_utc'] = pd.to_datetime(raw_weather_df['timestamp'], errors='coerce', utc=True)
raw_weather_df['ts_eat'] = raw_weather_df['ts_utc'].dt.tz_convert('Africa/Addis_Ababa').dt.tz_localize(None)

utc_hourly = raw_weather_df.groupby(raw_weather_df['ts_utc'].dt.hour)['temp_c'].mean()
eat_hourly = raw_weather_df.groupby(raw_weather_df['ts_eat'].dt.hour)['temp_c'].mean()

print(f"Diurnal temperature peak on raw UTC clock: Hour {utc_hourly.idxmax()}:00 UTC")
print(f"Diurnal temperature peak on converted EAT: Hour {eat_hourly.idxmax()}:00 EAT (Physical Solar Peak)")

## A3 & A4. Multi-Table Integration & Join Audit
We run the full pipeline to integrate Trips, Weather, and Events tables.

In [ ]:
trips_clean = load_ride_demand("../data/raw/ride_demand_train.csv")
weather_clean = load_weather("../data/raw/weather_hourly.csv")
events_clean = load_events("../data/raw/events_calendar.csv")

# Join sequence
master = add_weather_features(trips_clean, weather_clean)
master = add_event_features(master, events_clean)
master = add_calendar_features(master)

print(f"\nRow count before joins : {len(trips_clean)}")
print(f"Row count after joins  : {len(master)} (Row count invariant preserved)")
print(f"Master feature columns : {len(master.columns)}")

## A5. Join Proof: Tracing 3 Specific Zone-Hours
Verifying join correctness across rain, event window, and public holiday cases:

In [ ]:
rain_sample = master[(master['rain_mm'] > 5.0) & (master['zone'] == 'BOLE')].iloc[0]
event_sample = master[(master['event_active_window'] == 1.0) & (master['event_football_match'] == 1.0)].iloc[0]
holiday_sample = master[(master['event_public_holiday'] == 1.0) & (master['pickup_hour'].astype(str).str.startswith('2025-01-07'))].iloc[0]

print("1. Rainy Hour Sample:")
print(f"   Zone: {rain_sample['zone']} | Hour: {rain_sample['pickup_hour']} | Rain: {rain_sample['rain_mm']}mm | Rain_3h: {rain_sample['rain_3h']}mm | Trips: {rain_sample['trips']}")

print("\n2. Event Window Sample:")
print(f"   Zone: {event_sample['zone']} | Hour: {event_sample['pickup_hour']} | Football: {event_sample['event_football_match']} | Active Window: {event_sample['event_active_window']} | Trips: {event_sample['trips']}")

print("\n3. Public Holiday Sample (Genna):")
print(f"   Zone: {holiday_sample['zone']} | Hour: {holiday_sample['pickup_hour']} | Holiday Flag: {holiday_sample['event_public_holiday']} | Trips: {holiday_sample['trips']}")

## A6. Feature Engineering Table
All 28 features in the final model are known at forecast time.

In [ ]:
feat_dict = pd.read_csv("../data/processed/data_dictionary_master.csv")
display(feat_dict.head(15))

## A7. Automated Integrity Checks
Executing the 6 automated validation assertions to guarantee dataset cleanliness and leakage freedom:

In [ ]:
checks = []
canonical = set(CANONICAL_ZONES)

c1 = set(master['zone'].unique()) == canonical
checks.append(('1. Zone Domain Integrity (Exactly 12 Canonical Zones)', 'PASS' if c1 else 'FAIL'))

c2 = not master.duplicated(subset=['zone', 'pickup_hour']).any()
checks.append(('2. Unique Zone-Hour Keys (No Duplicates)', 'PASS' if c2 else 'FAIL'))

c3 = master['trips'].notna().all() and (master['trips'] >= 0).all()
checks.append(('3. Target Validity (Trips >= 0 and Non-Null)', 'PASS' if c3 else 'FAIL'))

c4 = not any(c in master.columns for c in ['avg_fare_birr', 'avg_wait_min', 'active_drivers'])
checks.append(('4. Leakage Freedom (Operational Columns Dropped)', 'PASS' if c4 else 'FAIL'))

c5 = master['temp_c'].notna().all() and master['rain_mm'].notna().all()
checks.append(('5. Weather Feature Completeness', 'PASS' if c5 else 'FAIL'))

c6 = pd.to_datetime(master['pickup_hour']).min() >= pd.Timestamp('2025-01-01')
checks.append(('6. Time Horizon Alignment (2025 EAT)', 'PASS' if c6 else 'FAIL'))

c7 = master.drop(columns=['trips']).isna().sum().sum() == 0
checks.append(('7. Zero Missing Values in Master Features (Classical Model Ingestion Safe)', 'PASS' if c7 else 'FAIL'))

for name, status in checks:
    print(f'[{status}] {name}')

# Classical Model Ingestion Verification
from sklearn.linear_model import Ridge
X_test = pd.get_dummies(master.drop(columns=['trips', 'pickup_hour', 'record_id']), columns=['zone'], drop_first=True)
y_test = master['trips']
ridge = Ridge().fit(X_test, y_test)
print(f'Classical Ridge fit succeeded: R2 = {ridge.score(X_test, y_test):.3f}')


## A8. Master Tables Export
The integrated dataset is exported to `data/processed/master_train.csv` and documented in `data/processed/data_dictionary_master.csv`.

In [ ]:
out_path = Path("../data/processed/master_train.csv")
master.to_csv(out_path, index=False)
print(f"Master train table exported to: {out_path.resolve()}")
print(f"Exported row count: {len(master)} | Columns: {len(master.columns)}")